In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

from similarity import l2_normalize, manhattan_similarity, cosine_similarity
from scipy.stats import pearsonr, spearmanr
from keras.models import load_model
from dataset import create_dataset
from loader import load_splits
from settings import get_settings
from model import SiameseLSTM
import matplotlib.pyplot as plt
from keras import Model
import tensorflow as tf
import numpy as np
from paths import (
    vectorizer_path,
    siamese_path,
    embedding_path,
    history_path,
    processed_dir
)


In [2]:
settings = get_settings()


In [ ]:
splits = {
	"test": processed_dir
}

datasets = load_splits(splits)

test_df = datasets["test"]


In [6]:
embedding_matrix = np.load(embedding_path)

vectorizer_model = load_model(vectorizer_path)
vectorizer = vectorizer_model.layers[0]


In [ ]:
test_dataset = create_dataset(test_df, vectorizer, settings.batch_size)


In [ ]:
custom_objects={
	"SiameseLSTM": SiameseLSTM,
	"l2_normalize": l2_normalize,
	"manhattan_similarity": manhattan_similarity,
	"cosine_similarity": cosine_similarity,
}

model = load_model(
	siamese_path,
	custom_objects=custom_objects,
	compile=False
)

model.embedding.set_weights([embedding_matrix])

head_model = model.get_head_model()

history_dict = np.load(history_path, allow_pickle=True).item()


In [ ]:
def plot_history(history_dict):
	plt.figure(figsize=(18,5))

	# ---- Loss plot ----
	plt.subplot(1,3,1)
	plt.plot(history_dict["loss"], label="train")
	plt.plot(history_dict["val_loss"], label="validation")

	best_val_loss = np.min(history_dict["val_loss"])
	best_epoch_loss = np.argmin(history_dict["val_loss"])
	plt.scatter(best_epoch_loss, best_val_loss, color='red', s=100, label=f"best val_loss: {best_val_loss:.6f}")
	plt.title("Loss")
	plt.xlabel("Epoch")
	plt.ylabel("MSE Loss")
	plt.legend()

	# ---- MAE plot ----
	plt.subplot(1,3,2)
	plt.plot(history_dict["mae"], label="train")
	plt.plot(history_dict["val_mae"], label="validation")

	best_val_mae = np.min(history_dict["val_mae"])
	best_epoch_mae = np.argmin(history_dict["val_mae"])
	plt.scatter(best_epoch_mae, best_val_mae, color='red', s=100, label=f"best val_mae: {best_val_mae:.6f}")
	plt.title("MAE")
	plt.xlabel("Epoch")
	plt.ylabel("Mean Absolute Error")
	plt.legend()

	plt.tight_layout()
	plt.show()
	
	print(
		f"Best Val Loss (MSE): {best_val_loss:.6f} (epoch {best_epoch_loss + 1}) | "
		f"Best Val MAE: {best_val_mae:.6f} (epoch {best_epoch_mae + 1})"
	)

plot_history(history_dict)


In [ ]:
def evaluate_model(model: Model, dataset: tf.data.Dataset):
	y_true = np.concatenate([y.numpy() for _, y in dataset], axis=0)
	
	y_pred = model.predict(dataset).flatten()
	
	# Comprueba si la relación entre las predicciones y las etiquetas sigue una línea recta.
	# 1.0 --> correlación positiva perfecta
	# 0.0 --> sin correlación
	# -1.0 --> correlación negativa perfecta
	pearson, _ = pearsonr(y_true, y_pred)
	# Compara el orden de las predicciones y los valores reales.
	spearman, _ = spearmanr(y_true, y_pred)
	mse = np.mean((y_true - y_pred) ** 2)
	mae = np.mean(np.abs(y_true - y_pred))
	rmse = np.sqrt(mse)

	ss_res = np.sum((y_true - y_pred) ** 2)
	ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
	r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0.0

	print(
		f"Pearson: {pearson:.4f} | "
		f"Spearman: {spearman:.4f} | "
		f"MSE: {mse:.6f} | "
		f"MAE: {mae:.6f} | "
		f"RMSE: {rmse:.6f} | "
		f"R²: {r2:.4f} | "
	)

evaluate_model(model, test_dataset)


In [ ]:
sentences = [
	"Me encanta programar en Python.",
	"El perro corre rápido por el parque."
]

sentence_tensor = vectorizer(sentences)
print("Sentence tensor shape:", sentence_tensor.shape)

vecs = head_model(sentence_tensor)
print("Embeddings shape:", vecs.shape)

for i, sentence in enumerate(sentences):
	print(f"\nSentence {i+1}: '{sentence}'")
	print("Sentence tensor:")
	print(sentence_tensor[i])

sim = cosine_similarity(vecs[0:1], vecs[0:1])
print("Cosine similarity:", sim.item())
